Пример кода DAG

In [ ]:
from airflow import DAG
from airflow.operators.python_operator import PythonOperator
from datetime import datetime

default_args = {
    'owner': 'airflow',
    'start_date': datetime(2024, 9, 1),
    'retries': 1,
}

def read_data(**kwargs):
    with open('/path/to/input/file.csv', 'r') as file:
        data = file.read()
    return data

def process_data(**kwargs):
    data = kwargs['ti'].xcom_pull(task_ids='read_data_task')
    # Преобразование данных из CSV в JSON
    processed_data = convert_csv_to_json(data)
    return processed_data

def save_data(**kwargs):
    data = kwargs['ti'].xcom_pull(task_ids='process_data_task')
    with open('/path/to/output/file.json', 'w') as file:
        file.write(data)

with DAG('simple_dag', default_args=default_args, schedule_interval='@daily') as dag:
    read_data_task = PythonOperator(task_id='read_data_task', python_callable=read_data)
    process_data_task = PythonOperator(task_id='process_data_task', python_callable=process_data, provide_context=True)
    save_data_task = PythonOperator(task_id='save_data_task', python_callable=save_data, provide_context=True)

    read_data_task >> process_data_task >> save_data_task
